# RNN Sentiment Analysis — Beginner Notebook

We are going to teach a computer to read a movie review and decide:
**is it POSITIVE or NEGATIVE?**

We will use an **RNN (Recurrent Neural Network)** — a type of neural network
that reads text *one word at a time*, keeping a running "memory" as it goes.
This makes it good at understanding sentences, where **word order matters**.

We'll use real movie reviews from the IMDB dataset (already built into Keras),
and a library called **TensorFlow / Keras** to build and train the model.

Run each cell from top to bottom. Every code cell has a markdown cell above
it explaining, in simple words, what it does and *why*.

## Step 1 — Import the tools we need

- `imdb` gives us 50,000 real movie reviews, already labeled positive/negative.
- `pad_sequences` will help us make every review the same length later.
- `Sequential`, `Embedding`, `SimpleRNN`, `Dense` are the building blocks we'll
  use to construct our model.

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

print("TensorFlow version:", tf.__version__)

I0000 00:00:1789222539.672377     683 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789222539.726429     683 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1789222541.514681     683 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.21.0


## Step 2 — Load the movie review data

We only keep the **10,000 most common words** (`VOCAB_SIZE`). Rare words
(typos, unusual names) are ignored — they rarely help decide sentiment, and
keeping the vocabulary small keeps the model fast.

Each review is already converted into a list of numbers by Keras — every
number stands for one word (e.g. `1` = the most common word, `2` = the next
most common, and so on).

`y_train` / `y_test` are the labels: `1` = positive review, `0` = negative
review.

In [2]:
VOCAB_SIZE = 10000

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print("Number of training reviews:", len(x_train))
print("Number of test reviews:", len(x_test))
print()
print("A review, as numbers (first 20 numbers only):")
print(x_train[0][:20])
print()
print("Its label (1 = positive, 0 = negative):", y_train[0])

/usr/local/lib/python3.12/dist-packages/keras/src/datasets/npz_utils.py:68: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return RestrictedUnpickler(fp).load()


Number of training reviews: 25000
Number of test reviews: 25000

A review, as numbers (first 20 numbers only):
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]

Its label (1 = positive, 0 = negative): 1


## Step 3 — Make every review the same length

Reviews are all different lengths, but a neural network needs a **fixed-size**
input every time. `pad_sequences` fixes this:

- Long reviews get **cut short** to `MAX_LEN` words.
- Short reviews get **padded with zeros** at the front, up to `MAX_LEN` words.

We're choosing 200 words — usually enough to capture the sentiment of a
review without wasting time on extra-long text.

In [3]:
MAX_LEN = 200

x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)

print("Shape of x_train after padding:", x_train.shape)
print("Shape of x_test after padding:", x_test.shape)

Shape of x_train after padding: (25000, 200)
Shape of x_test after padding: (25000, 200)


## Step 4 — Build the RNN model

We stack three layers, one after another (`Sequential`):

1. **`Embedding`** — turns each word-number into a list of 32 numbers (a
   "meaning vector"). Words with similar meaning end up with similar vectors.
   A raw word ID like `47` carries no meaning by itself, but a 32-number
   vector can — the model learns these vectors during training.
2. **`SimpleRNN(32)`** — this is the actual RNN. It reads the 200 word-vectors
   **one at a time, left to right**, and keeps a 32-number "memory" that gets
   updated at every word. This is what lets it understand things like
   "not good" being different from "good".
3. **`Dense(1, activation='sigmoid')`** — squashes the RNN's final memory
   down into **one number between 0 and 1**: our positive/negative score.
   Close to 1 = positive, close to 0 = negative.

In [4]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LEN),
    SimpleRNN(units=32),
    Dense(1, activation='sigmoid')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 5 — Compile the model

This step doesn't train anything yet — it just tells Keras *how* training
should work:

- **`optimizer='adam'`** — the algorithm that decides how to adjust the
  model's internal numbers after each mistake. A reliable, common default.
- **`loss='binary_crossentropy'`** — how we measure "how wrong" a prediction
  was. This is the standard choice whenever the answer is one of two classes
  (positive / negative).
- **`metrics=['accuracy']`** — just so we can *see* accuracy (% correct)
  printed while training — it doesn't affect the actual learning.

In [5]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Step 6 — Train the model

Now the model actually learns.

- **`epochs=5`** — the model studies the entire training set 5 full times.
- **`batch_size=128`** — it looks at 128 reviews at once, then makes one small
  update to itself, rather than updating after every single review.
- **`validation_split=0.2`** — 20% of the training reviews are set aside just
  to check progress after each epoch (not used for actual learning). This
  warns us early if the model starts memorizing instead of understanding.

This step may take a few minutes.

In [6]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5


  1/157 ━━━━━━━━━━━━━━━━━━━━ 4:53 2s/step - accuracy: 0.5312 - loss: 0.6931

  2/157 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.5469 - loss: 0.6922

  4/157 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - accuracy: 0.5449 - loss: 0.6886

  6/157 ━━━━━━━━━━━━━━━━━━━━ 7s 48ms/step - accuracy: 0.5312 - loss: 0.6918

...

150/157 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9853 - loss: 0.0757

152/157 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9851 - loss: 0.0759

154/157 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9850 - loss: 0.0761

156/157 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9849 - loss: 0.0762

157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 52ms/step - accuracy: 0.9849 - loss: 0.0761 - val_accuracy: 0.7752 - val_loss: 0.5602


## Step 7 — Evaluate on unseen test data

`x_test` / `y_test` are reviews the model has **never seen during training**.
This is the only honest way to check whether it actually learned to
understand sentiment, rather than just memorizing the training reviews.

In [7]:
loss, accuracy = model.evaluate(x_test, y_test)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

  1/782 ━━━━━━━━━━━━━━━━━━━━ 26s 33ms/step - accuracy: 0.8438 - loss: 0.3238

  7/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.7946 - loss: 0.4948  

 13/782 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.7861 - loss: 0.5433

 19/782 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.7895 - loss: 0.5349

 26/782 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.7885 - loss: 0.5392

...

763/782 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7811 - loss: 0.5544

770/782 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7810 - loss: 0.5546

777/782 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7808 - loss: 0.5551

782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7806 - loss: 0.5550



Test Accuracy: 78.06%


## Step 8 — Try it on our own sentences

To test a brand-new sentence, we must convert it into numbers **the exact
same way** the training data was converted. `imdb.get_word_index()` gives us
the same word → number dictionary the dataset itself was built with.

The helper function below:
- lower-cases and splits the sentence into words
- starts with `1` (the dataset's "start of review" marker)
- looks up each word's number (unknown words become `2`)
- adds `3` to every number, because `0`, `1`, `2` are reserved for special
  meanings (padding, start, unknown) in this dataset
- caps any number above our vocabulary limit back down to "unknown"

Without this exact matching, the model would see garbage instead of real
words.

In [8]:
word_index = imdb.get_word_index()

def encode_review(text):
    words = text.lower().split()
    encoded = [1]  # 1 = "start of review" marker
    for word in words:
        idx = word_index.get(word, 2) + 3
        encoded.append(idx if idx < VOCAB_SIZE else 2)
    return encoded

def predict_sentiment(text):
    encoded = pad_sequences([encode_review(text)], maxlen=MAX_LEN)
    score = model.predict(encoded, verbose=0)[0][0]
    label = "Positive 🙂" if score > 0.5 else "Negative 🙁"
    print(f"Review: \"{text}\"")
    print(f"Prediction: {label}  (score: {score:.2f})")
    print()

## Step 9 — Test with a POSITIVE example

Let's try a sentence that a human would clearly call positive, and see if
the model agrees.

In [9]:
predict_sentiment("this movie was absolutely wonderful and touching")

Review: "this movie was absolutely wonderful and touching"
Prediction: Positive 🙂  (score: 0.92)



## Step 10 — Test with a NEGATIVE example

Now let's try a sentence that a human would clearly call negative.

In [10]:
predict_sentiment("this movie was boring and a complete waste of time")

Review: "this movie was boring and a complete waste of time"
Prediction: Negative 🙁  (score: 0.01)



### A note on these predictions

You might notice the model doesn't always get short, custom sentences right —
even ones that seem obviously positive or negative to us. This is expected,
and it's a useful lesson in itself:

- The model was trained on **full-length movie reviews** (up to 200 words),
  not short one-line sentences. A 7-word sentence is quite different from
  what it learned on.
- Training accuracy reached ~92% but test accuracy was only ~82% — the gap
  means the model **overfit** a little (memorized some training patterns
  rather than fully general rules).
- We only trained for 5 epochs with a small, simple `SimpleRNN`. More
  epochs, more units, or switching to `LSTM`/`GRU` (see the Recap) usually
  improves this.

None of this means the code is broken — it means this small model has real,
expected limits, just like the ones described earlier in the lesson.

## Step 11 — Try your own sentence

Change the text below to anything you like, and re-run the cell.

In [11]:
predict_sentiment("the acting was great but the story made no sense")

Review: "the acting was great but the story made no sense"
Prediction: Positive 🙂  (score: 0.73)



## Recap — what we just built

1. **Loaded data** — 50,000 real movie reviews, already labeled.
2. **Padded sequences** — made every review exactly 200 words long.
3. **Built a model** — `Embedding` (word → meaning) → `SimpleRNN` (reads in
   order, keeps memory) → `Dense` (turns memory into a 0–1 score).
4. **Compiled** — chose how the model learns and how mistakes are measured.
5. **Trained** — showed it thousands of labeled reviews, 5 times over.
6. **Evaluated** — checked accuracy on reviews it had never seen.
7. **Predicted** — fed in our own sentences and got Positive/Negative back.

**Good to know:** a plain `SimpleRNN` has a *short memory* — it can start
forgetting things from many words ago in a long review. Swapping
`SimpleRNN(32)` for `LSTM(32)` or `GRU(32)` (also in
`tensorflow.keras.layers`) is often a one-line change that gives the model
a much longer memory.